# 阶段 3-4：表达式系统与 Alpha Pool

本 Notebook 使用小型 toy 数据演示当前实现的主链路：RPN 公式 -> 表达式树 -> 因子求值 -> IC/RankIC -> alpha pool。这里不接入 PPO、XGBoost、回测或任何交易接口。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "PROJECT_PLAN.md").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from quant_rl_alpha.alpha import AlphaPool, daily_ic, mean_ic, mean_rank_ic
from quant_rl_alpha.expression import evaluate, is_semantically_valid, parse_rpn, valid_actions

## 1. 构造 toy 行情和标签

toy 数据只用于检查工程口径。真实研究时应使用阶段 1-2 生成的 `daily_panel.parquet`、`universe_monthly.parquet` 和 `labels.parquet`，并按训练/验证/测试日期切分。

In [ ]:
dates = pd.bdate_range("2024-01-01", periods=60)
rows = []
for symbol, base, slope in [("000001", 10.0, 0.30), ("000002", 12.0, 0.15), ("000003", 9.0, 0.45)]:
    for i, date in enumerate(dates):
        close = base + slope * i
        volume = 1000 + i * 20 + int(symbol[-1]) * 50
        rows.append({
            "date": date,
            "symbol": symbol,
            "open": close - 0.05,
            "high": close + 0.30,
            "low": close - 0.30,
            "close": close,
            "volume": volume,
            "amount": volume * close,
            "vwap": close,
        })

daily_panel = pd.DataFrame(rows)
labels = daily_panel[["date", "symbol", "close"]].copy()
labels["future_20d_return"] = labels.groupby("symbol")["close"].shift(-20) / labels["close"] - 1
labels = labels.dropna(subset=["future_20d_return"])[["date", "symbol", "future_20d_return"]]

daily_panel.head(), labels.head()

## 2. RPN 解析、渲染和 action mask

时序算子的 RPN 形式是 `表达式 时间窗口 算子`，例如 `close 20d Mean`。`SEP` 只在当前栈刚好形成一个非纯常数表达式时允许。

In [ ]:
tokens = ["BEG", "close", "20d", "Mean", "close", "Div", "1", "Sub", "SEP"]
expr = parse_rpn(tokens)
print(expr.to_formula())
print(valid_actions(["BEG", "close", "20d"])[:10])

## 3. 因子求值和语义检查

Evaluator 只负责按公式生成 raw factor value。除零、`Log` 非正、全 NaN、截面常数等语义问题会被标记为不可用，后续 RL 环境可据此给 `-1` 奖励。

In [ ]:
factor = evaluate(expr, daily_panel, value_name="alpha_1")
print(is_semantically_valid(factor, value_name="alpha_1"))
factor.dropna().head()

## 4. IC、RankIC 和 Alpha Pool

阶段 4 使用每日截面相关衡量单因子和组合表现。Pool loss 使用论文口径：每日截面去均值并做 L2 归一化，不是普通 z-score。

In [ ]:
factor_for_metrics = factor.rename(columns={"alpha_1": "value"})
print(daily_ic(factor_for_metrics, labels).dropna().head())
print("mean_ic", mean_ic(factor_for_metrics, labels))
print("mean_rank_ic", mean_rank_ic(factor_for_metrics, labels))

pool = AlphaPool(labels, capacity=3, l1_alpha=0.0)
result_1 = pool.add("alpha_1", expr.to_formula(), factor_for_metrics)

expr_2 = parse_rpn(["BEG", "close", "20d", "Delta", "SEP"])
factor_2 = evaluate(expr_2, daily_panel)
result_2 = pool.add("alpha_2", expr_2.to_formula(), factor_2)

print(result_1)
print(result_2)
print(pool.weight_map())